In [1]:
!pip install kaggle

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!kaggle datasets list

ref                                                                      title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
algozee/teenager-menthal-healy                                           Social Media Impact on Teen Mental Health                16190  2026-04-05 08:04:21.823000          32432        677                1  
laveshjadon/ai-impact-on-students                                        Impact of Ai on Students                               1187170  2026-05-10 23:12:10.070000           2496         58                1  
abdulmaliklodhra/social-media-addiction-and-mental-health-dataset        Social Media Addiction & Mental Health Dataset        13975225  2026-05-22 10:15:52.543000 

In [3]:
!kaggle datasets download -d lakshmi25npathi/online-retail-dataset

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset
License(s): other
online-retail-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
import os
import zipfile

zip_file_path = "online-retail-dataset.zip"
extraction_target = "data"

if os.path.exists(zip_file_path):
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_target)
        print(f"Successfully extracted {zip_file_path} to the '{extraction_target}' folder.")
    except zipfile.BadZipFile:
        print(f"Error: The file '{zip_file_path}' is corrupted. Please re-download.")

else:
    print(f"Error: '{zip_file_path}' not found. Check your Kaggle download step.")
    

Successfully extracted online-retail-dataset.zip to the 'data' folder.


##Load into pandas

In [5]:
import pandas as pd

df = pd.read_excel("data/online_retail_II.xlsx")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
xls = pd.ExcelFile("data/online_retail_II.xlsx")
print(xls.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


Use Latest Year Sheet

In [7]:

df = pd.read_excel("data/online_retail_II.xlsx", sheet_name='Year 2010-2011')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Data Understanding

In [8]:
df.shape

(541910, 8)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [10]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [11]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,541910.000000,541910,541910.000000,406830.000000
mean,9.552234,2011-07-04 13:35:22.342307584,4.611138,15287.684160
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.080957,NaN,96.759765,1713.603074


## Standardize Column Names

In [12]:
df.columns = df.columns.str.strip().str.replace(' ','_')
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer_ID', 'Country'],
      dtype='object')

## Remove Missing Customer ID & Description Null

In [13]:
df = df.dropna(subset=['Customer_ID', 'Description'])

df['Customer_ID'] = df['Customer_ID'].astype('Int64')

In [15]:
## Validate Cancellations 
canceled_invoices = df['Invoice'].astype(str).str.startswith('C')
all_canceled_have_negative_qty = df[canceled_invoices]['Quantity'].max() <= 0
outliers = df[(df['Quantity'] < 0) & (~canceled_invoices)]

print(f"Are all cancellations caught by Quantity filter?: {all_canceled_have_negative_qty}")
print(f"Negative quantities missing 'C' prefix: {len(outliers)}")

Are all cancellations caught by Quantity filter?: True
Negative quantities missing 'C' prefix: 0


## Remove Invalid Transactions

In [16]:
# Remove negative or zero quantity & zero or negative price
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

In [17]:
df['Customer_ID'] = df['Customer_ID'].astype(int)

## Create Revenue Column

In [18]:
df['TotalPrice'] = df['Quantity']*df['Price']

## Final Validation

In [19]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 397885 entries, 0 to 541909
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      397885 non-null  object        
 1   StockCode    397885 non-null  object        
 2   Description  397885 non-null  object        
 3   Quantity     397885 non-null  int64         
 4   InvoiceDate  397885 non-null  datetime64[ns]
 5   Price        397885 non-null  float64       
 6   Customer_ID  397885 non-null  int64         
 7   Country      397885 non-null  object        
 8   TotalPrice   397885 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 30.4+ MB


,Quantity,InvoiceDate,Price,Customer_ID,TotalPrice
count,397885.000000,397885,397885.000000,397885.000000,397885.000000
mean,12.988208,2011-07-10 23:41:56.419316992,3.116525,15294.416882,22.396989
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000,0.001000
25%,2.000000,2011-04-07 11:12:00,1.250000,13969.000000,4.680000
50%,6.000000,2011-07-31 14:39:00,1.950000,15159.000000,11.800000
75%,12.000000,2011-10-20 14:33:00,3.750000,16795.000000,19.800000
max,80995.000000,2011-12-09 12:50:00,8142.750000,18287.000000,168469.600000
std,179.331551,NaN,22.097861,1713.144421,309.070653


In [20]:
df.shape

(397885, 9)

##PHASE 5- PUSH DATA TO SQL SERVER

In [21]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'PostgreSQL ODBC Driver(ANSI)', 'PostgreSQL ODBC Driver(UNICODE)']


## Create Connection

In [22]:
from sqlalchemy import create_engine
server = r"localhost\SQLEXPRESS"


engine = create_engine(
    f"mssql+pyodbc://@{server}/OnlineRetailDB?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server",
    fast_executemany=True
)

In [23]:
## Optimize Data Types

from sqlalchemy import  types

dtype_mapping = {
    'Invoice': types.String(20),
    'Stockcode': types.String(20),
    'Description': types.String(255),
    'Quantity': types.Integer(),
    'InvoiceDate': types.DateTime(),
    'Price': types.Numeric(10,2),
    'Customer_ID': types.Integer(),
    'Country': types.String(50),
    'TotalPrice': types.Numeric(12,2)
}
                               


In [24]:
## Push Data

df.to_sql(
    name="OnlineRetail",
    con=engine,
    if_exists='replace',
    index=False,
    dtype=dtype_mapping,
    chunksize=10000   # Important for performance
)
    

-40